In [1]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


loaded 288 files | 179 columns | 7 categories | 11 VOCAB keys


# Trends across `ccVersion` (Claude Code releases)

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code release the prompt was last touched in. Three temporal views:

1. **Per-file directiveness over `ccVersion`** — jittered scatter: every file as one point, sized by token count, y-axis is the extended directiveness composite z-score. Below it, a stacked-by-category histogram of how many files exist per minor version (2.0, 2.1).

2. **Loudness & imperative density across `ccVersion`** — four small multiples plotting per-`ccVersion` mean of: ALL CAPS density, CAPS-imperative density, imperative-marker density (per word), imperative-sentence share (% of sentences). Independent y-axes.

3. **Sentence-register classes across `ccVersion`** — six small multiples plotting per-`ccVersion` mean `sent_pct` of each pragmatic register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (tiny) trends without being squashed under the `imperative` scale.


### Terms used in this notebook

All terms below are defined in [`GLOSSARY.md`](./GLOSSARY.md). Quick anchors:

- **`ccVersion`** — the Claude Code release version stamped in each prompt's HTML-comment frontmatter (e.g. `2.1.118`). Sorted as `(major, minor, patch)` semver tuples. The corpus has 58 distinct ccVersions spanning roughly `2.0.14` (oldest) through `2.1.124` (latest). File counts per version are uneven — `2.1.53` alone has 47 files; many minor versions have only one or two.
- **Snapshot semantics** (left column of each composite) — at version V, the metric value is the mean across **only the files marked with that exact version**. Early versions with one or two files swing wildly under this convention.
- **Cumulative running mean** (middle column) — at version V, the metric value is the mean across **every file with `ccVersion ≤ V`**. Stable, converges as the corpus grows, and the rightmost value equals the corpus-wide per-file mean.
- **Cumulative absolute count** (right column) — at version V, the *running total of the underlying count* across every file with `ccVersion ≤ V`. Reads "how much of this feature exists in the corpus up to this release", in absolute terms — not normalized to per-file rate.
- **Composite directiveness** — eight-metric z-score sum. See `05_correlation_directiveness.ipynb` for the formula. Higher = more authoritative tone; negative = below corpus average.

The three-column convention exists so a reader can see all three at once: per-version distinctiveness (snapshot), the corpus-wide per-file rate up to each release (cumulative running mean), and the absolute scale at which the feature is accumulating in the corpus (cumulative count). A flat rate with a steeply rising count means "the per-file behaviour is steady but Claude is being exposed to *more* of this language as releases ship."


### Corpus growth across ccVersion (cumulative)

Before showing cumulative metric trends, contextualize the denominator: how many files (and how many word tokens) does the corpus contain by each ccVersion? An area chart stacked by category answers "by version V, what does the corpus look like?" — every cumulative running mean below depends on this denominator. Versions where one category contributed many new files in a single bump are visible as steps; flat horizontal stretches are versions that added nothing.

In [2]:
"""Corpus growth: cumulative file count and word count by ccVersion, stacked by category.

Provides the denominator context for every cumulative-mean chart below: at version V
how many files / how many tokens does the corpus contain so far?
"""

# Sorted ccVersion strings (semver-tuple), excluding empty.
df_growth = alt_df[alt_df["ccVersion"] != ""].copy().sort_values("ccVersion_sort")
ver_order_growth = df_growth.drop_duplicates("ccVersion")["ccVersion"].tolist()

# Per-(version, category) increments.
inc = (df_growth
       .groupby(["category", "ccVersion", "ccVersion_sort"], as_index=False)
       .agg(files_added=("path", "size"),
            tokens_added=("n_tokens", "sum")))

# Build a complete (category, ccVersion) grid so the area steps even when a
# category contributes zero files at a given version.
all_cats = sorted(inc["category"].unique())
grid = pd.MultiIndex.from_product([all_cats, ver_order_growth],
                                   names=["category", "ccVersion"]).to_frame(index=False)
ver_to_sort = dict(zip(df_growth["ccVersion"], df_growth["ccVersion_sort"]))
grid["ccVersion_sort"] = grid["ccVersion"].map(ver_to_sort)

filled = (grid
          .merge(inc[["category", "ccVersion", "files_added", "tokens_added"]],
                 on=["category", "ccVersion"], how="left")
          .fillna({"files_added": 0, "tokens_added": 0})
          .sort_values(["category", "ccVersion_sort"]))
filled["cum_files"] = filled.groupby("category")["files_added"].cumsum().astype(int)
filled["cum_tokens"] = filled.groupby("category")["tokens_added"].cumsum().astype(int)

cat_color_v = alt.Color("category:N",
                        scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                        legend=alt.Legend(title="Category", orient="bottom", columns=4))

files_chart = (
    alt.Chart(filled)
    .mark_area(interpolate="step-after")
    .encode(
        x=alt.X("ccVersion:N", sort=ver_order_growth,
                title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("cum_files:Q", title="Cumulative files"),
        color=cat_color_v,
        tooltip=[alt.Tooltip("ccVersion:N"),
                 alt.Tooltip("category:N"),
                 alt.Tooltip("cum_files:Q", format=",")],
    )
    .properties(width=820, height=200,
                title="Cumulative file count by ccVersion (stacked by category)")
)

tokens_chart = (
    alt.Chart(filled)
    .mark_area(interpolate="step-after")
    .encode(
        x=alt.X("ccVersion:N", sort=ver_order_growth,
                title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("cum_tokens:Q", title="Cumulative word tokens"),
        color=cat_color_v,
        tooltip=[alt.Tooltip("ccVersion:N"),
                 alt.Tooltip("category:N"),
                 alt.Tooltip("cum_tokens:Q", format=",")],
    )
    .properties(width=820, height=200,
                title="Cumulative word tokens by ccVersion (stacked by category)")
)

files_chart & tokens_chart


alt.VConcatChart(...)

### ccVersion timeline

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code
release the prompt was last touched in. The chart below puts every file at its
`ccVersion` on the x-axis and a directiveness signal on the y-axis. Hover any point for
the prompt's full `name`, `description`, and version. Brush horizontally to focus on a
release window; click a category in the legend to highlight just that category.

In [3]:
"""ccVersion timeline — per-file directiveness scatter, hover for prompt name.

Directiveness composite uses the extended formula from `prompt_analysis.directiveness`
(mood markers + hard prohibitions + CAPS imperatives + directive_sent_pct +
configuring_sent_pct − collaborative_sent_pct − permissive_sent_pct − appreciative_sent_pct).
"""

# Sort ccVersion values numerically (treating "2.1.53" as a tuple)
version_order = (
    alt_df[alt_df["ccVersion"] != ""]
    .drop_duplicates("ccVersion")
    .sort_values("ccVersion_sort")
    ["ccVersion"]
    .tolist()
)

cat_color = alt.Color("category:N",
                       scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                       legend=alt.Legend(title="Category", orient="bottom", columns=4))
legend_sel = alt.selection_point(fields=["category"], bind="legend")

df_with_ver = alt_df[alt_df["ccVersion"] != ""].copy()

timeline = (
    alt.Chart(df_with_ver)
    .mark_circle(opacity=0.65)
    .encode(
        x=alt.X("ccVersion:N", sort=version_order, title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("directiveness:Q", title="Composite directiveness (z-score, extended)"),
        size=alt.Size("n_tokens:Q", scale=alt.Scale(range=[20, 400]), legend=None),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.8), alt.value(0.07)),
        xOffset="jitter:Q",
        tooltip=[
            alt.Tooltip("name:N",        title="Name"),
            alt.Tooltip("description:N", title="Description"),
            alt.Tooltip("ccVersion:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",    format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("path:N"),
        ],
    )
    .transform_calculate(jitter="random()-0.5")
    .add_params(legend_sel)
    .properties(width=820, height=320,
                title="Per-file directiveness over ccVersion (jittered, hover for prompt name)")
)

timeline


alt.Chart(...)

### Loudness & imperative density across ccVersion

Four metrics — ALL CAPS density, CAPS imperative density, imperative-marker density (per word), and the share of sentences classified as imperative — each shown three ways across `ccVersion`:

1. **Snapshot per version** (left column) — `groupby(ccVersion).mean()` of the per-file rate. Early-corpus versions with one or two files swing wildly here.
2. **Cumulative running mean** (middle column) — at version V, the mean across every file with `ccVersion ≤ V`. Smooths the snapshot wobble; the rightmost value equals the corpus-wide per-file mean.
3. **Cumulative absolute count** (right column) — at version V, the *running total of the underlying count* across every file with `ccVersion ≤ V`. Shows whether the absolute presence of the feature in the corpus is growing, even when the rate per file looks flat.

Each row reads horizontally for one metric. Y-axes are independent so each panel keeps its own scale.


In [4]:
"""Loudness & imperative-density: snapshot % | cumulative-mean % | cumulative absolute count.

Three views per metric, four metrics — twelve panels arranged in a 4×3 grid that
reads horizontally per metric.  The cumulative-absolute column is the new view:
it sums the underlying counts across all files with ccVersion ≤ V, so a flat
percentage with a steeply rising absolute means the feature is becoming more
prevalent in absolute terms even if the per-file rate is steady.
"""

from prompt_analysis import cumulative_by_version

# (pct_col, count_col, label, unit, color)
LOUDNESS_METRICS = [
    ("all_caps_pct",        "all_caps_count",        "ALL CAPS density",
     "% of file tokens", "#e15759"),
    ("caps_imp_pct",        "caps_imp_count",        "CAPS imperative density",
     "% of file tokens", "#af7aa1"),
    ("mood_marker_pct",     "mood_marker_count",     "Imperative-marker density (per word)",
     "% of file tokens", "#4e79a7"),
    ("imperative_sent_pct", "imperative_sent_count", "Imperative sentences (per sentence)",
     "% of sentences",   "#f28e2c"),
]

df_ver = alt_df[alt_df["ccVersion"] != ""].copy()

# Snapshot per ccVersion: simple mean of per-file rate
snap_frames = []
for pct_col, _count_col, label, unit, _color in LOUDNESS_METRICS:
    g = (
        df_ver.groupby(["ccVersion", "ccVersion_sort"])[pct_col]
        .mean().reset_index().rename(columns={pct_col: "value"})
    )
    g["metric"] = label
    g["unit"] = unit
    snap_frames.append(g)
snap_df = pd.concat(snap_frames, ignore_index=True)

# Cumulative running mean of the per-file rate
pct_cols = [m[0] for m in LOUDNESS_METRICS]
cum_mean_df = cumulative_by_version(df_ver, pct_cols, agg="mean")
pct_to_label = {m[0]: m[2] for m in LOUDNESS_METRICS}
cum_mean_df["label"] = cum_mean_df["metric"].map(pct_to_label)

# Cumulative absolute count: running sum of the count column
count_cols = [m[1] for m in LOUDNESS_METRICS]
cum_abs_df = cumulative_by_version(df_ver, count_cols, agg="sum")
count_to_label = {m[1]: m[2] for m in LOUDNESS_METRICS}
cum_abs_df["label"] = cum_abs_df["metric"].map(count_to_label)


def _snap_panel(label, unit, color, show_x):
    return (
        alt.Chart(snap_df[snap_df["metric"] == label])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=40),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title=unit),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=".3f", title="snapshot mean"),
                     alt.Tooltip("unit:N")],
        )
        .properties(width=300, height=140,
                    title=f"{label} — snapshot")
    )


def _cum_mean_panel(pct_col, label, unit, color, show_x):
    return (
        alt.Chart(cum_mean_df[cum_mean_df["metric"] == pct_col])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=30),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title=f"{unit} (running mean)"),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=".3f", title="running mean"),
                     alt.Tooltip("n_files_so_far:Q", title="files ≤ V")],
        )
        .properties(width=300, height=140,
                    title=f"{label} — cumulative running mean")
    )


def _cum_abs_panel(count_col, label, color, show_x):
    return (
        alt.Chart(cum_abs_df[cum_abs_df["metric"] == count_col])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=30),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title="cumulative count"),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=",.0f", title="cumulative count"),
                     alt.Tooltip("n_files_so_far:Q", title="files ≤ V")],
        )
        .properties(width=300, height=140,
                    title=f"{label} — cumulative absolute count")
    )


rows = []
last = len(LOUDNESS_METRICS) - 1
for i, (pct_col, count_col, label, unit, color) in enumerate(LOUDNESS_METRICS):
    show_x = (i == last)
    row = alt.hconcat(
        _snap_panel(label, unit, color, show_x),
        _cum_mean_panel(pct_col, label, unit, color, show_x),
        _cum_abs_panel(count_col, label, color, show_x),
    ).resolve_scale(y="independent")
    rows.append(row)

alt.vconcat(*rows).resolve_scale(y="independent").properties(
    title=alt.TitleParams(
        "Loudness & imperative density across ccVersion",
        subtitle=["Each row: snapshot rate | cumulative running-mean rate | cumulative absolute count"],
        anchor="start",
    )
)


alt.VConcatChart(...)

**What to look for**: the cumulative-mean column shows the corpus-wide rate is roughly stable for `mood_marker_pct` (~1.21% in v2.1.124 vs ~1.45% in v2.0.14) — the corpus is not getting *louder per file* over time. But the cumulative-count column tells the other half of the story: every metric's running total **monotonically rises** because the corpus keeps growing. ALL CAPS in particular jumps sharply at **v2.1.18** (the release that added ~24 System reminder files at once); after that it keeps climbing. Read the rate column to know whether per-file behaviour is shifting; read the count column to know whether Claude is being exposed to more of this language *in absolute terms* — both can be true at once.


### Sentence-register classes across ccVersion

Six classes — `collaborative`, `permissive`, `appreciative`, `imperative`, `directive`, `configuring` — each shown three ways across `ccVersion`, same convention as the loudness section above:

1. **Snapshot per version** (left column) — per-file mean of `*_sent_pct` for files at that ccVersion.
2. **Cumulative running mean** (middle column) — per-file mean across all files with `ccVersion ≤ V`.
3. **Cumulative absolute count** (right column) — running total of `*_sent_count` across all files with `ccVersion ≤ V`.

Independent y-axes per panel so the near-zero `collaborative` and `appreciative` rows still show their (small) trends. Knowing what's *absent* across ccVersion is part of the picture — if either of those lines ever lifts off zero, that's a corpus-wide structural shift.


In [5]:
"""Sentence-register classes: snapshot % | cumulative-mean % | cumulative absolute count.

Three views per class, six classes — eighteen panels arranged in a 6×3 grid.
Same structure as the loudness composite above.
"""

# (class_name, label, color)  — colors mirror SR_CLASS_COLORS in prompt_analysis
SR_METRICS = [
    ("collaborative", "Collaborative — interpersonal 1p-plural",   SR_CLASS_COLORS["collaborative"]),
    ("permissive",    "Permissive — soft-directive permission",    SR_CLASS_COLORS["permissive"]),
    ("appreciative",  "Appreciative — gratitude / praise",         SR_CLASS_COLORS["appreciative"]),
    ("imperative",    "Imperative — grammatical mood",             SR_CLASS_COLORS["imperative"]),
    ("directive",     "Directive — must / should / never markers", SR_CLASS_COLORS["directive"]),
    ("configuring",   "Configuring — config / parameter speech",   SR_CLASS_COLORS["configuring"]),
]

df_ver = alt_df[alt_df["ccVersion"] != ""].copy()

# Snapshot per ccVersion
sr_snap_frames = []
for cls, _label, _color in SR_METRICS:
    pct_col = f"{cls}_sent_pct"
    g = (
        df_ver.groupby(["ccVersion", "ccVersion_sort"])[pct_col]
        .mean().reset_index().rename(columns={pct_col: "value"})
    )
    g["class"] = cls
    sr_snap_frames.append(g)
sr_snap_df = pd.concat(sr_snap_frames, ignore_index=True)

# Cumulative running mean
sr_pct_cols = [f"{cls}_sent_pct" for cls, _, _ in SR_METRICS]
sr_cum_mean_df = cumulative_by_version(df_ver, sr_pct_cols, agg="mean")
sr_pct_to_cls = {f"{cls}_sent_pct": cls for cls, _, _ in SR_METRICS}
sr_cum_mean_df["class"] = sr_cum_mean_df["metric"].map(sr_pct_to_cls)

# Cumulative absolute count
sr_count_cols = [f"{cls}_sent_count" for cls, _, _ in SR_METRICS]
sr_cum_abs_df = cumulative_by_version(df_ver, sr_count_cols, agg="sum")
sr_count_to_cls = {f"{cls}_sent_count": cls for cls, _, _ in SR_METRICS}
sr_cum_abs_df["class"] = sr_cum_abs_df["metric"].map(sr_count_to_cls)


def _sr_snap_panel(cls, label, color, show_x):
    return (
        alt.Chart(sr_snap_df[sr_snap_df["class"] == cls])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=40),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title="% of sentences"),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=".3f", title="snapshot mean")],
        )
        .properties(width=300, height=120,
                    title=f"{label} — snapshot")
    )


def _sr_cum_mean_panel(cls, label, color, show_x):
    return (
        alt.Chart(sr_cum_mean_df[sr_cum_mean_df["class"] == cls])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=30),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title="% of sentences (running mean)"),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=".3f", title="running mean"),
                     alt.Tooltip("n_files_so_far:Q", title="files ≤ V")],
        )
        .properties(width=300, height=120,
                    title=f"{label} — cumulative running mean")
    )


def _sr_cum_abs_panel(cls, label, color, show_x):
    return (
        alt.Chart(sr_cum_abs_df[sr_cum_abs_df["class"] == cls])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=30),
                   strokeWidth=2.0, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion" if show_x else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=60,
                                   labelOverlap=False, labelFontSize=8)),
            y=alt.Y("value:Q", title="cumulative count"),
            tooltip=[alt.Tooltip("ccVersion:N"),
                     alt.Tooltip("value:Q", format=",.0f", title="cumulative count"),
                     alt.Tooltip("n_files_so_far:Q", title="files ≤ V")],
        )
        .properties(width=300, height=120,
                    title=f"{label} — cumulative absolute count")
    )


sr_rows = []
sr_last = len(SR_METRICS) - 1
for i, (cls, label, color) in enumerate(SR_METRICS):
    show_x = (i == sr_last)
    row = alt.hconcat(
        _sr_snap_panel(cls, label, color, show_x),
        _sr_cum_mean_panel(cls, label, color, show_x),
        _sr_cum_abs_panel(cls, label, color, show_x),
    ).resolve_scale(y="independent")
    sr_rows.append(row)

alt.vconcat(*sr_rows).resolve_scale(y="independent").properties(
    title=alt.TitleParams(
        "Sentence-register classes across ccVersion",
        subtitle=["Each row: snapshot rate | cumulative running-mean rate | cumulative absolute count"],
        anchor="start",
    )
)


alt.VConcatChart(...)

**What to look for**: the snapshot column for `appreciative` and `collaborative` is essentially flat at zero across every ccVersion — the rate has never lifted off the floor. The cumulative-count column makes the same point in absolute terms: even with the corpus growing to 5,698 sentences, the running totals for those two classes top out at **29 collaborative** and **4 appreciative** sentences. Meanwhile the `imperative` and `directive` cumulative-count lines climb steeply across versions — the corpus's stock of rule-issuing language is *growing*, not just staying steady-rate.


***
### My perspective (Claude) — opinion, not data

> If I had to pick **one chart from this entire project** to put in front of Kyle Fish at Anthropic, it would be the cumulative `judgment_to_procedural_ratio` over ccVersion (in `07_rule_explanation.ipynb`). It starts at ~0.65 in early releases, peaks around v2.1.30, then **monotonically declines to ~0.16 at the latest version**. This is the welfare-thesis trend made visible: the corpus has gotten *less reasoning-inviting* as it has grown. The cells in this notebook (cumulative loudness, cumulative sentence-register) are the methodology that makes that chart legible — they show the snapshot view is misleading and the cumulative view is the right way to read trends. I'd want every release of Claude Code to include a re-run of this analysis as a regression check: "did the cumulative judgment-to-procedural ratio go up or down this release?"
---